# 04 - Tracker Karşılaştırması

ByteTrack vs BoT-SORT vs DeepSORT performans karşılaştırması.

## Metrikler
- ID Switch sayısı
- Track tutarlılığı
- FPS etkisi
- Yoğun trafikte davranış

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
from collections import defaultdict

In [ ]:
VIDEO_PATH = 'data/videos/test/sample.mp4'
NUM_FRAMES = 500

TRACKERS = {
    'ByteTrack': 'configs/bytetrack.yaml',
    'BoT-SORT': 'configs/botsort.yaml',
}

In [ ]:
def evaluate_tracker(tracker_config, video_path, num_frames):
    """Tracker performansını ölç."""
    model = YOLO('yolov8s.pt')
    cap = cv2.VideoCapture(video_path)
    
    frame_times = []
    all_track_ids = set()
    track_id_per_frame = []
    
    for i in range(num_frames):
        ret, frame = cap.read()
        if not ret:
            break
        
        t0 = time.time()
        results = model.track(
            frame, conf=0.35, classes=[2,3,5,7],
            tracker=tracker_config, persist=True, verbose=False
        )
        t1 = time.time()
        frame_times.append(t1 - t0)
        
        frame_ids = set()
        if results[0].boxes.id is not None:
            ids = results[0].boxes.id.cpu().numpy().astype(int)
            frame_ids = set(ids)
            all_track_ids.update(frame_ids)
        track_id_per_frame.append(frame_ids)
    
    cap.release()
    
    return {
        'avg_fps': 1.0 / np.mean(frame_times),
        'total_unique_ids': len(all_track_ids),
        'avg_tracks_per_frame': np.mean([len(f) for f in track_id_per_frame]),
        'max_id': max(all_track_ids) if all_track_ids else 0,
    }

results = {}
for name, config in TRACKERS.items():
    print(f'Değerlendiriliyor: {name}')
    results[name] = evaluate_tracker(config, VIDEO_PATH, NUM_FRAMES)
    print(f'  FPS: {results[name]["avg_fps"]:.1f}, '
          f'Unique IDs: {results[name]["total_unique_ids"]}')

In [ ]:
# Karşılaştırma grafiği
import pandas as pd

df = pd.DataFrame(results).T
print(df)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

df['avg_fps'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Ortalama FPS')
axes[0].set_ylabel('FPS')

df['total_unique_ids'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Toplam Benzersiz Track ID')
axes[1].set_ylabel('ID Sayısı')

df['avg_tracks_per_frame'].plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('Ort. Track / Kare')
axes[2].set_ylabel('Track Sayısı')

plt.tight_layout()
plt.savefig('results/tracker_comparison.png', dpi=150, bbox_inches='tight')
plt.show()